In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Current MPI resolution follows the explicit native merge relation. No
# fuzzy name matching or invented identity merge is permitted.
patients = spark.table("raw_clinical_patient")
merges = spark.table("raw_clinical_patient_merge").filter("is_deleted = 0")
assert merges.groupBy("duplicate_id").count().filter("count > 1").count() == 0
mapping = patients.select(F.col("id").alias("patient_id"), F.col("id").alias("mpi_patient_id"))
# Resolve transitive merges with bounded convergence and fail on cycles.
seen = set()
for _ in range(100):
    next_map = mapping.alias("p").join(merges.alias("m"), F.col("p.mpi_patient_id") == F.col("m.duplicate_id"), "left").select(F.col("p.patient_id"), F.coalesce(F.col("m.survivor_id"), F.col("p.mpi_patient_id")).alias("mpi_patient_id"))
    changes = next_map.alias("n").join(mapping.alias("p"), "patient_id").filter(F.col("n.mpi_patient_id") != F.col("p.mpi_patient_id")).count()
    mapping = next_map.localCheckpoint(eager=True)
    if changes == 0: break
else: raise ValueError("MPI relation did not converge; inspect cyclic merges")
resolved = patients.join(mapping, patients.id == mapping.patient_id, "inner").drop("patient_id")
resolved.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("stg_patient")
